# RoadVision AI — Road Damage Semantic Segmentation
## Transfer Learning Pipeline · DeepLabV3-ResNet50 · Kaggle Cloud GPU

This notebook fine-tunes a **DeepLabV3 with ResNet-50 backbone** (pretrained on
COCO/VOC) for 3-class road damage segmentation:

| Class | ID | Colour |
|---|---|---|
| Background | 0 | Black |
| Pothole | 1 | Red `#FF3B30` |
| Crack | 2 | Amber `#FFCC00` |

### Dataset
**Road Damage Dataset: Potholes, Cracks, and Manholes**
(`lorenzoarcioni/road-damage-dataset-potholes-cracks-and-manholes`)

```
/kaggle/input/<dataset-slug>/
├── images/          ← JPEG road photographs
└── labels/          ← polygon TXT annotations  (used ✓)
```

Annotation format (`labels/*.txt`):
```
class_id  x1 y1  x2 y2  x3 y3  x4 y4   (normalized 0-1)
```

| Dataset class | Meaning  | Model class |
|---|---|---|
| 0 | Pothole  | **1** |
| 1 | Crack    | **2** |
| 2 | Manhole  | SKIPPED (background) |

### Why Transfer Learning?
ResNet-50 pretrained on ImageNet already learns rich low-level (edges, textures)
and mid-level (shapes) features that transfer directly to road surface analysis.
Fine-tuning from these weights converges faster and to a better optimum than
training U-Net from scratch — especially with the polygon-annotated dataset of
moderate size used here.

### Key Pipeline Features
- DeepLabV3-ResNet50 pretrained backbone, fine-tuned with differential LRs
- U-Net (scratch) preserved as optional baseline in a dedicated cell
- Automatic dataset discovery & image ↔ label pairing
- 70 / 15 / 15 split (no data leakage)
- Synchronized augmentations + ImageNet normalization
- Class-wise IoU & Dice · AMP · Checkpoint resume
- FastAPI-ready artifact export


---
## 1. Environment & Hardware Verification


In [ ]:
import os, sys, time, json, random, shutil, zipfile, warnings
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import cv2
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
import torchvision

warnings.filterwarnings("ignore", category=UserWarning)

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = True

set_seed(42)

print(f"Python      : {sys.version.split()[0]}")
print(f"PyTorch     : {torch.__version__}")
print(f"Torchvision : {torchvision.__version__}")
print(f"OpenCV      : {cv2.__version__}")
print("-" * 55)

if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"CUDA    : YES  |  GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"cuDNN   : {torch.backends.cudnn.version()}")
else:
    device = torch.device("cpu")
    print("CUDA    : NO (CPU only)")
    print("IMPORTANT: Enable GPU in Kaggle -> Settings -> Accelerator -> GPU T4 x2 or P100")


---
## 2. Configuration & Hyperparameters

> **Transfer learning LR strategy**: The ResNet-50 backbone is fine-tuned at a
> low learning rate (`backbone_lr = 1e-5`) to preserve learned ImageNet features.
> The DeepLabV3 segmentation head (randomly initialized for 3 classes) is trained
> at a 10x higher rate (`head_lr = 1e-4`) for faster convergence.


In [ ]:
CONFIG = {
    # -- Experiment -------------------------------------------------------
    "project_name"       : "RoadVision-AI",
    "experiment_id"      : "deeplabv3_resnet50_finetune_v1",
    "dataset_name"       : "Road Damage Dataset: Potholes Cracks and Manholes",

    # -- Paths ------------------------------------------------------------
    "kaggle_input_dir"   : "/kaggle/input",
    "output_dir"         : "/kaggle/working/roadvision_artifacts",
    "checkpoint_dir"     : "/kaggle/working/checkpoints",

    # -- Model architecture -----------------------------------------------
    "model_architecture" : "DeepLabV3_ResNet50",
    "backbone"           : "resnet50",
    "pretrained_weights" : "DeepLabV3_ResNet50_Weights.COCO_WITH_VOC_LABELS_V1",
    "in_channels"        : 3,
    "num_classes"        : 3,          # 0: Background  1: Pothole  2: Crack
    "aux_loss"           : True,       # DeepLabV3 auxiliary classifier supervision
    "aux_loss_weight"    : 0.4,        # Auxiliary loss weighting factor in training loss

    # -- Preprocessing (ImageNet stats — required for pretrained ResNet) ---
    "image_size"         : (512, 512), # (H, W) — must be divisible by 8
    "img_mean"           : [0.485, 0.456, 0.406],
    "img_std"            : [0.229, 0.224, 0.225],

    # -- Training ----------------------------------------------------------
    "batch_size"         : 8,          # Reduce to 4 if OOM; 16 on 16GB+ VRAM
    "num_epochs"         : 30,
    "backbone_lr"        : 1e-5,       # Fine-tune pretrained ResNet-50 slowly
    "head_lr"            : 1e-4,       # Train new segmentation head faster
    "weight_decay"       : 1e-4,
    "use_amp"            : True,       # Automatic Mixed Precision

    # -- Loss --------------------------------------------------------------
    "dice_loss_weight"   : 0.6,
    "ce_loss_weight"     : 0.4,
    "class_weights"      : [0.2, 1.5, 1.5],  # bg / pothole / crack

    # -- LR Scheduler ------------------------------------------------------
    "scheduler_type"     : "cosine",
    "min_lr"             : 1e-6,

    # -- Data split --------------------------------------------------------
    "train_ratio"        : 0.70,
    "val_ratio"          : 0.15,
    "test_ratio"         : 0.15,
    "seed"               : 42,
}

os.makedirs(CONFIG["output_dir"], exist_ok=True)
os.makedirs(CONFIG["checkpoint_dir"], exist_ok=True)

print("Configuration loaded:")
for k, v in CONFIG.items():
    print(f"  {k:<24}: {v}")


---
## 3. Semantic Class Definitions & Colour Palette

| ID | Name | Colour |
|---|---|---|
| 0 | Background | Black `#000000` |
| 1 | Pothole | Red `#FF3B30` |
| 2 | Crack | Amber `#FFCC00` |

> **Dataset note**: The original dataset contains a third class — Manholes
> (dataset class ID 2). Manholes are **excluded** from the model's damage
> classes and treated as background (class 0).


In [ ]:
CLASS_INFO = {
    0: {"id": 0, "name": "background", "label": "Road Surface / Background",
        "color_rgb": [  0,   0,   0], "color_hex": "#000000"},
    1: {"id": 1, "name": "pothole",    "label": "Pothole",
        "color_rgb": [255,  59,  48], "color_hex": "#FF3B30"},
    2: {"id": 2, "name": "crack",      "label": "Road Crack",
        "color_rgb": [255, 204,   0], "color_hex": "#FFCC00"},
}

ID_TO_NAME = {cls_id: info["name"] for cls_id, info in CLASS_INFO.items()}
NAME_TO_ID = {info["name"]: cls_id for cls_id, info in CLASS_INFO.items()}

# Original dataset class -> model class mapping
# dataset 0 (Pothole) -> 1, dataset 1 (Crack) -> 2, dataset 2 (Manhole) -> 0 (skip)
DATASET_TO_MODEL_CLASS = {0: 1, 1: 2, 2: 0}

def mask_to_color_image(mask: np.ndarray) -> np.ndarray:
    """Convert integer class mask (H,W) -> RGB image (H,W,3)."""
    out = np.zeros((*mask.shape, 3), dtype=np.uint8)
    for cls_id, info in CLASS_INFO.items():
        out[mask == cls_id] = info["color_rgb"]
    return out

def overlay_mask_on_image(img_rgb: np.ndarray, mask: np.ndarray,
                           alpha: float = 0.5) -> np.ndarray:
    """Alpha-blend coloured mask onto original image for damage pixels only."""
    color_mask = mask_to_color_image(mask)
    out = img_rgb.copy()
    dmg = mask > 0
    out[dmg] = ((1 - alpha) * img_rgb[dmg] + alpha * color_mask[dmg]).astype(np.uint8)
    return out

print("Class info:")
for cls_id, info in CLASS_INFO.items():
    print(f"  ID {cls_id}: {info['name']:12s} | {info['label']} | RGB {info['color_rgb']}")
print("\nDataset-to-model class map:", DATASET_TO_MODEL_CLASS)


---
## 4. Dataset Directory Inspection

We scan the Kaggle input directory to locate:
- `images/` folder containing JPEG road photographs
- `labels/` folder containing polygon `.txt` annotation files

Each `.txt` file has the same stem as its corresponding image.


In [ ]:
def inspect_dataset_root(kaggle_input: str) -> Tuple[Optional[Path], Optional[Path]]:
    """
    Walk /kaggle/input looking for an images/ + labels/ pair.
    Returns (images_dir, labels_dir) or (None, None).
    """
    root = Path(kaggle_input)
    print(f"Scanning: {root.resolve()}")

    # Print top-level structure
    for item in sorted(root.iterdir()):
        print(f"  {'[DIR] ' if item.is_dir() else '[FILE]'} {item.name}")
        if item.is_dir():
            for sub in sorted(item.iterdir())[:8]:
                print(f"      {'[DIR] ' if sub.is_dir() else '[FILE]'} {sub.name}")

    # Strategy 1: direct images/ and labels/ at dataset level
    for dataset_dir in [root] + sorted([p for p in root.iterdir() if p.is_dir()]):
        images_dir = dataset_dir / "images"
        labels_dir = dataset_dir / "labels"
        if images_dir.exists() and labels_dir.exists():
            img_count = sum(1 for f in images_dir.iterdir()
                            if f.suffix.lower() in {".jpg", ".jpeg", ".png"})
            lbl_count  = sum(1 for f in labels_dir.iterdir() if f.suffix == ".txt")
            print(f"\n[FOUND] Dataset directory : {dataset_dir}")
            print(f"        images/  -> {img_count} image files")
            print(f"        labels/  -> {lbl_count} annotation files")
            if img_count > 0:
                return images_dir, labels_dir

    print("\n[WARNING] Could not auto-detect images/ + labels/ directories.")
    print("Please check that the dataset is attached via '+ Add Data'.")
    return None, None

images_dir, labels_dir = inspect_dataset_root(CONFIG["kaggle_input_dir"])


---
## 5. Annotation File Inspection

We read a few sample `.txt` files to verify the polygon format and class IDs
actually present in the dataset.

Expected format per line:
```
class_id  x1 y1  x2 y2  x3 y3  x4 y4   (normalized 0-1)
```


In [ ]:
def inspect_annotation_samples(labels_dir: Path, n_samples: int = 5) -> Dict:
    """
    Read n random annotation files, print content, and collect class frequency.
    Returns: {raw_class_id: count_of_polygon_instances}
    """
    if labels_dir is None:
        print("No labels directory found.")
        return {}

    all_txt = sorted(labels_dir.glob("*.txt"))
    if not all_txt:
        print("No .txt files found in labels/")
        return {}

    print(f"Total annotation files: {len(all_txt)}")
    sample_files = random.sample(all_txt, min(n_samples, len(all_txt)))

    class_counts: Dict[int, int] = {}

    # Inspect samples
    print("\n--- Sample annotation files ---")
    for txt_path in sample_files:
        with open(txt_path, "r") as f:
            content = f.read().strip()
        print(f"\n  File: {txt_path.name}")
        if not content:
            print("    (empty file)")
        else:
            for line in content.splitlines()[:5]:
                parts = line.split()
                if parts:
                    raw_cls = int(float(parts[0]))
                    n_coords = len(parts) - 1
                    n_points = n_coords // 2
                    print(f"    class={raw_cls}  coords={n_coords} values ({n_points} points)")

    # Full frequency scan
    print("\n--- Scanning ALL annotation files for class frequency ---")
    total_empty = 0
    for txt_path in all_txt:
        with open(txt_path, "r") as f:
            lines = f.read().strip().splitlines()
        if not lines:
            total_empty += 1
            continue
        for line in lines:
            parts = line.split()
            if parts:
                raw_cls = int(float(parts[0]))
                class_counts[raw_cls] = class_counts.get(raw_cls, 0) + 1

    print(f"  Empty annotation files: {total_empty}")
    print("  Class instance counts (raw dataset IDs):")
    dataset_class_names = {0: "Pothole", 1: "Crack", 2: "Manhole"}
    for cls_id in sorted(class_counts):
        name = dataset_class_names.get(cls_id, f"Unknown-{cls_id}")
        model_cls = DATASET_TO_MODEL_CLASS.get(cls_id, 0)
        note = "SKIPPED (background)" if model_cls == 0 else f"-> model class {model_cls}"
        print(f"    Dataset class {cls_id} ({name:8s}): {class_counts[cls_id]:6d} polygons  {note}")

    missing = []
    if 0 not in class_counts:
        missing.append("Pothole (dataset class 0)")
    if 1 not in class_counts:
        missing.append("Crack (dataset class 1)")
    if missing:
        print(f"\n[WARNING] Missing classes in dataset: {missing}")
    else:
        print("\n[OK] Both Pothole (class 0) and Crack (class 1) are present in the dataset.")

    return class_counts

class_frequency = inspect_annotation_samples(labels_dir)


---
## 6. Image ↔ Label File Pairing

Match every image in `images/` with its corresponding `.txt` file in `labels/`
by stem name. Report any unmatched files.


In [ ]:
def build_paired_samples(images_dir: Optional[Path],
                         labels_dir: Optional[Path]) -> List[Dict]:
    """
    Pair image files with their polygon annotation .txt files.
    Returns list of dicts: {image_path, anno_path, anno_type}.
    """
    if images_dir is None or labels_dir is None:
        print("[!] images_dir or labels_dir is None. No pairs built.")
        return []

    IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp"}
    all_images = [p for p in images_dir.iterdir()
                  if p.suffix.lower() in IMAGE_EXTS]
    all_labels = {p.stem: p for p in labels_dir.glob("*.txt")}

    paired, unmatched_images = [], []

    for img_path in sorted(all_images):
        if img_path.stem in all_labels:
            paired.append({
                "image_path": str(img_path),
                "anno_path" : str(all_labels[img_path.stem]),
                "anno_type" : "polygon_txt",
            })
        else:
            unmatched_images.append(img_path.name)

    matched_stems  = {Path(p["image_path"]).stem for p in paired}
    unmatched_labs = [s for s in all_labels if s not in matched_stems]

    print(f"Images found     : {len(all_images)}")
    print(f"Label files found: {len(all_labels)}")
    print(f"Matched pairs    : {len(paired)}")
    if unmatched_images:
        print(f"Unmatched images : {len(unmatched_images)} (e.g. {unmatched_images[:3]})")
    if unmatched_labs:
        print(f"Unmatched labels : {len(unmatched_labs)} (e.g. {unmatched_labs[:3]})")
    if paired:
        print(f"\nSample pair:")
        print(f"  Image: {paired[0]['image_path']}")
        print(f"  Label: {paired[0]['anno_path']}")
    return paired

paired_samples = build_paired_samples(images_dir, labels_dir)


---
## 7. Polygon-to-Mask Rasterisation

The `decode_annotation` function:
1. Reads the JPEG image with `cv2.imread`.
2. Opens the paired `.txt` annotation file.
3. For each line, reads `class_id` and normalized `(x, y)` coordinate pairs.
4. Maps dataset class IDs to model class IDs:
   - `0 -> 1` (Pothole)
   - `1 -> 2` (Crack)
   - `2 -> 0` (Manhole -> skipped)
5. Converts normalized coordinates to pixel coordinates.
6. Rasterises the polygon onto the mask canvas using **`cv2.fillPoly`**.
7. Resizes image and mask to `CONFIG["image_size"]`.

> **Annotation limitation**: These polygon masks are **approximate**.
> Crack polygons are drawn as filled quadrilateral hulls; actual crack width
> is typically narrower than the polygon boundary suggests.


In [ ]:
def decode_annotation(sample_info: Dict,
                      target_hw: Tuple[int, int] = None) -> Tuple[np.ndarray, np.ndarray]:
    """
    Load image + decode polygon TXT annotations into a 3-class integer mask.

    Returns
    -------
    image_rgb : np.ndarray  shape (H, W, 3)  dtype uint8
    mask      : np.ndarray  shape (H, W)     dtype int64
                values: 0=Background  1=Pothole  2=Crack
    """
    img_bgr = cv2.imread(sample_info["image_path"])
    if img_bgr is None:
        raise ValueError(f"Cannot read image: {sample_info['image_path']}")
    image_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    h, w = image_rgb.shape[:2]

    mask = np.zeros((h, w), dtype=np.int64)
    anno_path = sample_info["anno_path"]

    if os.path.exists(anno_path) and os.path.getsize(anno_path) > 0:
        with open(anno_path, "r") as f:
            lines = f.read().strip().splitlines()

        for line in lines:
            parts = line.strip().split()
            if len(parts) < 5:
                continue

            raw_cls   = int(float(parts[0]))
            model_cls = DATASET_TO_MODEL_CLASS.get(raw_cls, 0)

            if model_cls == 0:
                continue

            coords   = [float(v) for v in parts[1:]]
            poly_pts = []
            for i in range(0, len(coords) - 1, 2):
                px = int(np.clip(coords[i]     * w, 0, w - 1))
                py = int(np.clip(coords[i + 1] * h, 0, h - 1))
                poly_pts.append([px, py])

            if len(poly_pts) < 3:
                continue

            pts_arr = np.array([poly_pts], dtype=np.int32)
            cv2.fillPoly(mask, pts_arr, color=int(model_cls))

    if target_hw is not None:
        th, tw = target_hw
        image_rgb = cv2.resize(image_rgb, (tw, th), interpolation=cv2.INTER_LINEAR)
        mask = cv2.resize(
            mask.astype(np.uint8), (tw, th), interpolation=cv2.INTER_NEAREST
        ).astype(np.int64)

    return image_rgb, mask

print("decode_annotation() defined.")
print("Dataset class -> model class:", DATASET_TO_MODEL_CLASS)


---
## 8. Visualise Images, Generated Masks & Overlays

We display 6 randomly selected samples to visually confirm:
- Polygon masks are rasterised correctly
- Pothole (Red) and Crack (Amber/Yellow) regions appear plausible
- Empty-annotation images show an all-black mask (no damage labelled)

> These masks are derived from polygon vertices, not dense pixel annotations.


In [ ]:
def visualize_samples(samples: List[Dict], n: int = 6,
                      target_hw: Tuple[int, int] = (384, 384)) -> None:
    if not samples:
        print("No samples to visualise.")
        return

    n = min(n, len(samples))
    indices = random.sample(range(len(samples)), n)

    fig, axes = plt.subplots(n, 3, figsize=(15, 4 * n))
    if n == 1:
        axes = np.expand_dims(axes, 0)

    fig.suptitle(
        "Road Damage Dataset: Potholes, Cracks, and Manholes\n"
        "Red = Pothole (model class 1)  |  Yellow = Crack (model class 2)  |  "
        "Manholes are EXCLUDED",
        fontsize=11, y=1.01
    )

    legend_patches = [
        mpatches.Patch(color=[c/255 for c in CLASS_INFO[1]["color_rgb"]], label="Pothole"),
        mpatches.Patch(color=[c/255 for c in CLASS_INFO[2]["color_rgb"]], label="Crack"),
        mpatches.Patch(color="gray", label="Manhole (excluded)"),
    ]

    for row, idx in enumerate(indices):
        img_rgb, mask = decode_annotation(samples[idx], target_hw=target_hw)
        color_mask  = mask_to_color_image(mask)
        overlay_img = overlay_mask_on_image(img_rgb, mask, alpha=0.55)

        pothole_px = int(np.sum(mask == 1))
        crack_px   = int(np.sum(mask == 2))
        img_name   = Path(samples[idx]["image_path"]).name

        axes[row, 0].imshow(img_rgb)
        axes[row, 0].set_title(f"{img_name}", fontsize=8)
        axes[row, 0].axis("off")

        axes[row, 1].imshow(color_mask)
        axes[row, 1].set_title(
            f"Polygon Mask  |  Pothole: {pothole_px}px  Crack: {crack_px}px", fontsize=8)
        axes[row, 1].axis("off")

        axes[row, 2].imshow(overlay_img)
        axes[row, 2].set_title("Overlay (approximate polygon masks)", fontsize=8)
        axes[row, 2].axis("off")

    fig.legend(handles=legend_patches, loc="lower center", ncol=3,
               bbox_to_anchor=(0.5, -0.01), fontsize=9)
    plt.tight_layout()
    plt.show()

if paired_samples:
    visualize_samples(paired_samples, n=6)
else:
    print("Visualisation skipped: no paired samples found.")


---
## 9. Class-Presence Audit Before Training

Before we train, verify that **both pothole and crack pixels actually appear
in the generated masks**. If either class is absent, training would be
meaningless for that class.


In [ ]:
def audit_class_coverage(samples: List[Dict],
                          max_inspect: int = 200) -> Dict[int, int]:
    """
    Sample up to max_inspect files, decode masks, accumulate pixel counts.
    Returns {model_class_id: total_pixel_count}.
    """
    if not samples:
        return {}

    subset = random.sample(samples, min(max_inspect, len(samples)))
    pixel_counts = {0: 0, 1: 0, 2: 0}

    for s in tqdm(subset, desc="Auditing class coverage", leave=False):
        _, mask = decode_annotation(s)
        for cls_id in pixel_counts:
            pixel_counts[cls_id] += int(np.sum(mask == cls_id))

    total_px = sum(pixel_counts.values())
    print(f"\nClass-pixel audit over {len(subset)} sampled images:")
    for cls_id, count in pixel_counts.items():
        pct = 100 * count / total_px if total_px else 0
        name = ID_TO_NAME[cls_id]
        flag = ""
        if cls_id > 0 and count == 0:
            flag = " <-- WARNING: class absent!"
        print(f"  Class {cls_id} ({name:12s}): {count:>12,} px  ({pct:5.2f}%){flag}")

    if pixel_counts[1] == 0:
        print("\n[ERROR] Pothole pixels = 0. Check class mapping / annotation files.")
    if pixel_counts[2] == 0:
        print("\n[WARNING] Crack pixels = 0 in sampled subset. "
              "Cracks may be sparse or absent in this dataset.")
    if pixel_counts[1] > 0 and pixel_counts[2] > 0:
        print("\n[OK] Both Pothole and Crack classes are present in the masks.")

    return pixel_counts

if paired_samples:
    pixel_audit = audit_class_coverage(paired_samples)
else:
    print("Audit skipped: no samples found.")
    pixel_audit = {}


---
## 10. Dataset Splitting & Augmentation Pipeline

The full paired sample list is **shuffled deterministically** using the fixed
seed then split at the image level (no data leakage):

| Split | Ratio | Purpose |
|---|---|---|
| Train | 70% | Gradient updates |
| Validation | 15% | Checkpoint selection |
| Test | 15% | Final unbiased evaluation |

Augmentations applied **only to the training split** (synchronised on image +
mask so spatial transforms are identical):
- Random Horizontal Flip (p=0.5)
- Random Vertical Flip (p=0.2)
- Random Brightness/Contrast Jitter (p=0.5, image only)

> **Preprocessing**: ImageNet mean and std normalization is **required** for the
> pretrained ResNet-50 backbone to function correctly, and is identical to what
> `torchvision.models.segmentation.DeepLabV3_ResNet50_Weights` expects.


In [ ]:
class RoadVisionDataset(Dataset):
    """
    PyTorch Dataset for RoadVision Semantic Segmentation.
    Polygon masks are decoded on-the-fly from TXT annotation files.
    Applies ImageNet normalization — required for pretrained ResNet backbone.
    """
    def __init__(self, samples: List[Dict],
                 image_size: Tuple[int, int],
                 is_train: bool = True):
        self.samples    = samples
        self.image_size = image_size
        self.is_train   = is_train
        self.mean = np.array(CONFIG["img_mean"], dtype=np.float32)
        self.std  = np.array(CONFIG["img_std"],  dtype=np.float32)

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        image_rgb, mask = decode_annotation(
            self.samples[idx], target_hw=self.image_size)

        if self.is_train:
            # Horizontal flip
            if random.random() > 0.5:
                image_rgb = np.fliplr(image_rgb)
                mask      = np.fliplr(mask)
            # Vertical flip
            if random.random() > 0.8:
                image_rgb = np.flipud(image_rgb)
                mask      = np.flipud(mask)
            # Brightness / Contrast jitter (image only)
            if random.random() > 0.5:
                alpha = random.uniform(0.85, 1.15)
                beta  = random.uniform(-15, 15)
                image_rgb = np.clip(
                    alpha * image_rgb.astype(np.float32) + beta, 0, 255
                ).astype(np.uint8)

        # ImageNet normalisation (required for pretrained ResNet-50 backbone)
        norm = image_rgb.astype(np.float32) / 255.0
        norm = (norm - self.mean) / self.std
        img_tensor  = torch.from_numpy(norm.transpose(2, 0, 1)).float()
        mask_tensor = torch.from_numpy(mask.copy()).long()
        return img_tensor, mask_tensor


# ---- Split -------------------------------------------------------
if paired_samples:
    idx_all = list(range(len(paired_samples)))
    random.seed(CONFIG["seed"])
    random.shuffle(idx_all)

    n_total = len(idx_all)
    n_train = int(n_total * CONFIG["train_ratio"])
    n_val   = int(n_total * CONFIG["val_ratio"])

    train_samples = [paired_samples[i] for i in idx_all[:n_train]]
    val_samples   = [paired_samples[i] for i in idx_all[n_train:n_train + n_val]]
    test_samples  = [paired_samples[i] for i in idx_all[n_train + n_val:]]

    print("Data split:")
    print(f"  Train : {len(train_samples):5d} ({CONFIG['train_ratio']*100:.0f}%)")
    print(f"  Val   : {len(val_samples):5d} ({CONFIG['val_ratio']*100:.0f}%)")
    print(f"  Test  : {len(test_samples):5d} ({CONFIG['test_ratio']*100:.0f}%)")

    train_ds = RoadVisionDataset(train_samples, CONFIG["image_size"], is_train=True)
    val_ds   = RoadVisionDataset(val_samples,   CONFIG["image_size"], is_train=False)
    test_ds  = RoadVisionDataset(test_samples,  CONFIG["image_size"], is_train=False)

    nw = 2 if os.name != "nt" else 0
    train_loader = DataLoader(train_ds, CONFIG["batch_size"], shuffle=True,
                              num_workers=nw, pin_memory=True)
    val_loader   = DataLoader(val_ds,   CONFIG["batch_size"], shuffle=False,
                              num_workers=nw, pin_memory=True)
    test_loader  = DataLoader(test_ds,  CONFIG["batch_size"], shuffle=False,
                              num_workers=nw, pin_memory=True)

    print(f"\nDataLoaders ready (batch={CONFIG['batch_size']}):")
    print(f"  Train batches: {len(train_loader)}")
    print(f"  Val   batches: {len(val_loader)}")
    print(f"  Test  batches: {len(test_loader)}")
else:
    train_loader = val_loader = test_loader = None
    train_ds = val_ds = test_ds = None
    print("[!] No dataset found — DataLoaders not created.")


---
## 11. Primary Model: DeepLabV3-ResNet50 (Transfer Learning)

We load `torchvision`'s DeepLabV3 with ResNet-50 backbone, pretrained on
**COCO with VOC labels** (21 classes including road-relevant features).

### What changes for 3-class fine-tuning:
1. **Backbone (ResNet-50)**: Weights frozen at ImageNet/COCO values. Fine-tuned
   at a low learning rate (`backbone_lr = 1e-5`) to preserve generalised features.
2. **ASPP Classifier head** (`model.classifier`): The final `Conv2d` layer
   (`classifier[4]`) is **replaced** with a new `Conv2d(256, 3, kernel_size=1)`
   that outputs 3-class logits (Background, Pothole, Crack).
3. **Auxiliary classifier** (`model.aux_classifier`): The final layer is also
   replaced for safety. Auxiliary loss is **disabled** during fine-tuning to keep
   the training objective simple and stable.

### Why DeepLabV3 over U-Net for transfer learning?
- Atrous Spatial Pyramid Pooling (ASPP) captures multi-scale context — ideal for
  detecting both small cracks and large potholes in the same image.
- ResNet-50 backbone provides pretrained feature hierarchies that require far
  fewer epochs to converge than training U-Net from scratch.
- `torchvision` provides officially verified weights with documented preprocessing.


In [ ]:
from torchvision.models.segmentation import (
    deeplabv3_resnet50,
    DeepLabV3_ResNet50_Weights,
)

NUM_CLASSES = CONFIG["num_classes"]   # 3: background, pothole, crack

def build_deeplabv3(num_classes: int = 3, aux_loss: bool = True) -> nn.Module:
    """
    Load DeepLabV3-ResNet50 with COCO pretrained weights and replace the
    segmentation head to output `num_classes` logits per pixel.

    Architecture note
    -----------------
    model.backbone   : ResNet-50 pretrained feature extractor
    model.classifier : DeepLabHead (ASPP + Conv layers)
      └ classifier[4]: Conv2d(256, 21, 1)  <-- replaced with Conv2d(256, num_classes, 1)
    model.aux_classifier: AuxiliaryHead
      └ aux_classifier[4]: Conv2d(256, 21, 1) <-- also replaced, aux loss enabled with weight=0.4
    """
    weights = DeepLabV3_ResNet50_Weights.COCO_WITH_VOC_LABELS_V1
    model = deeplabv3_resnet50(weights=weights, aux_loss=aux_loss)

    # --- Replace main segmentation head -----------------------------------
    # DeepLabHead.classifier[4] is the final Conv2d(256, 21, kernel_size=1)
    in_features_main = model.classifier[4].in_channels   # 256
    model.classifier[4] = nn.Conv2d(
        in_features_main, num_classes, kernel_size=1)

    # --- Replace auxiliary head (keeps architecture consistent) -----------
    # AuxiliaryHead[-1] is Conv2d(256, 21, kernel_size=1)
    in_features_aux = model.aux_classifier[4].in_channels  # 256
    model.aux_classifier[4] = nn.Conv2d(
        in_features_aux, num_classes, kernel_size=1)

    return model


model = build_deeplabv3(num_classes=NUM_CLASSES, aux_loss=CONFIG["aux_loss"]).to(device)

# Parameter summary
backbone_params    = sum(p.numel() for p in model.backbone.parameters())
classifier_params  = sum(p.numel() for p in model.classifier.parameters())
aux_params         = sum(p.numel() for p in model.aux_classifier.parameters())
total_params       = sum(p.numel() for p in model.parameters())
trainable_params   = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("Model: DeepLabV3-ResNet50 (transfer learning)")
print(f"  Pretrained weights    : COCO_WITH_VOC_LABELS_V1")
print(f"  Backbone parameters   : {backbone_params:>12,}  (ResNet-50, fine-tuned @ lr={CONFIG['backbone_lr']:.0e})")
print(f"  Classifier parameters : {classifier_params:>12,}  (ASPP head, trained @ lr={CONFIG['head_lr']:.0e})")
print(f"  Aux-head parameters   : {aux_params:>12,}  (active during training, aux_weight=0.4)")
print(f"  Total parameters      : {total_params:>12,}")
print(f"  Trainable parameters  : {trainable_params:>12,}")

# Forward-pass sanity check
# DeepLabV3 returns an OrderedDict with key 'out' (and optional 'aux')
model.eval()
with torch.no_grad():
    _dummy = torch.randn(2, 3, *CONFIG["image_size"]).to(device)
    _out_dict = model(_dummy)
    _logits = _out_dict["out"]
    print(f"\nSanity check:")
    print(f"  Input  shape : {tuple(_dummy.shape)}")
    print(f"  Output keys  : {list(_out_dict.keys())}")
    print(f"  Logits shape : {tuple(_logits.shape)}")
    assert _logits.shape == (2, NUM_CLASSES, *CONFIG["image_size"]), \
        f"Expected (2, {NUM_CLASSES}, {CONFIG['image_size']}) got {tuple(_logits.shape)}"
print("Architecture verified.")
model.train()


---
## 11b. Optional Baseline: U-Net (Train from Scratch)

The original U-Net implementation is preserved below for reference and
comparison. To switch to U-Net as the active model, comment out the
DeepLabV3 block above and uncomment the instantiation line at the bottom
of this cell.

**U-Net is NOT the active model** in this notebook. DeepLabV3-ResNet50 above
is the primary pipeline.


In [ ]:
# ============================================================
# U-Net Baseline (optional — not the active model)
# ============================================================

class DoubleConv(nn.Module):
    """[Conv2d(3x3) -> BN -> ReLU] x 2"""
    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch,  out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.net(x)


class RoadVisionUNet(nn.Module):
    """U-Net for 3-class road damage segmentation (train from scratch)."""
    def __init__(self, in_channels: int = 3, num_classes: int = 3,
                 features: List[int] = None):
        super().__init__()
        if features is None:
            features = [64, 128, 256, 512]
        self.pool  = nn.MaxPool2d(2, 2)
        self.downs = nn.ModuleList()
        self.ups   = nn.ModuleList()

        ch = in_channels
        for f in features:
            self.downs.append(DoubleConv(ch, f))
            ch = f

        self.bottleneck = DoubleConv(features[-1], features[-1] * 2)

        for f in reversed(features):
            self.ups.append(nn.ConvTranspose2d(f * 2, f, 2, 2))
            self.ups.append(DoubleConv(f * 2, f))

        self.head = nn.Conv2d(features[0], num_classes, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Returns plain tensor (not dict) — compatible with training engine
        skips = []
        for down in self.downs:
            x = down(x)
            skips.append(x)
            x = self.pool(x)

        x = self.bottleneck(x)
        skips = skips[::-1]

        for i in range(0, len(self.ups), 2):
            x = self.ups[i](x)
            sk = skips[i // 2]
            if x.shape != sk.shape:
                x = F.interpolate(x, size=sk.shape[2:],
                                  mode="bilinear", align_corners=True)
            x = self.ups[i + 1](torch.cat([sk, x], dim=1))

        return self.head(x)


# -- To use U-Net as the active model instead of DeepLabV3, uncomment: --
# model = RoadVisionUNet(
#     in_channels=CONFIG["in_channels"],
#     num_classes=CONFIG["num_classes"],
# ).to(device)
# CONFIG["model_architecture"] = "UNet"
# print("Active model switched to: RoadVisionUNet (scratch)")

print("RoadVisionUNet class defined (optional baseline — not currently active).")
print("Active model: DeepLabV3-ResNet50 (defined in Cell 11).")


---
## 12. Hybrid Loss: Weighted Cross-Entropy + Multi-Class Dice

Road damage pixels are rare (<5% of all pixels). A plain Cross-Entropy
loss would trivially achieve >95% pixel accuracy by predicting all-background.

We use a combined loss:
$$\mathcal{L} = w_{\text{CE}} \cdot \mathcal{L}_{\text{WCE}} + w_{\text{Dice}} \cdot \mathcal{L}_{\text{Dice}}$$

Class weights `[0.2, 1.5, 1.5]` down-weight background and boost
the pothole and crack gradients proportionally.


In [ ]:
class MulticlassDiceLoss(nn.Module):
    """Soft multi-class Dice loss with Laplace smoothing."""
    def __init__(self, num_classes: int = 3, smooth: float = 1.0,
                 ignore_index: Optional[int] = None):
        super().__init__()
        self.num_classes  = num_classes
        self.smooth       = smooth
        self.ignore_index = ignore_index

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        probs = F.softmax(logits, dim=1)
        ohe   = F.one_hot(targets, self.num_classes).permute(0, 3, 1, 2).float()
        loss  = 0.0
        n     = 0
        for c in range(self.num_classes):
            if c == self.ignore_index:
                continue
            p  = probs[:, c].contiguous().view(-1)
            t  = ohe[:, c].contiguous().view(-1)
            inter = (p * t).sum()
            denom = p.sum() + t.sum()
            loss += 1.0 - (2.0 * inter + self.smooth) / (denom + self.smooth)
            n    += 1
        return loss / max(n, 1)


class RoadVisionLoss(nn.Module):
    """Weighted CE + Multi-class Dice hybrid loss."""
    def __init__(self, num_classes: int = 3,
                 class_weights: List[float] = None,
                 ce_w: float = 0.4, dice_w: float = 0.6):
        super().__init__()
        if class_weights is None:
            class_weights = [0.2, 1.5, 1.5]
        self.ce   = nn.CrossEntropyLoss(
            weight=torch.tensor(class_weights, dtype=torch.float32))
        self.dice = MulticlassDiceLoss(num_classes)
        self.ce_w = ce_w
        self.dice_w = dice_w

    def forward(self, logits, targets):
        if self.ce.weight.device != logits.device:
            self.ce.weight = self.ce.weight.to(logits.device)
        l_ce   = self.ce(logits, targets)
        l_dice = self.dice(logits, targets)
        return self.ce_w * l_ce + self.dice_w * l_dice, l_ce, l_dice


criterion = RoadVisionLoss(
    num_classes=CONFIG["num_classes"],
    class_weights=CONFIG["class_weights"],
    ce_w=CONFIG["ce_loss_weight"],
    dice_w=CONFIG["dice_loss_weight"],
)
print("Loss function initialised. Class weights:", CONFIG["class_weights"])


---
## 13. Evaluation Metrics: Class-wise IoU & Dice

$$\text{IoU}_c = \frac{\text{TP}_c}{\text{TP}_c + \text{FP}_c + \text{FN}_c}
\qquad
\text{Dice}_c = \frac{2\,\text{TP}_c}{2\,\text{TP}_c + \text{FP}_c + \text{FN}_c}$$

Computed from accumulated confusion matrix across the entire split.


In [ ]:
class MetricTracker:
    """Accumulates per-pixel confusion matrix for IoU and Dice computation."""
    def __init__(self, num_classes: int = 3):
        self.num_classes = num_classes
        self.reset()

    def reset(self):
        self.cm = np.zeros((self.num_classes, self.num_classes), dtype=np.int64)

    def update(self, preds: torch.Tensor, targets: torch.Tensor):
        p = preds.detach().cpu().numpy().flatten()
        t = targets.detach().cpu().numpy().flatten()
        valid = (t >= 0) & (t < self.num_classes)
        idx   = self.num_classes * t[valid] + p[valid]
        self.cm += np.bincount(idx, minlength=self.num_classes ** 2).reshape(
            self.num_classes, self.num_classes)

    def compute(self) -> Dict[str, float]:
        cm = self.cm
        tp = np.diag(cm)
        fp = cm.sum(0) - tp
        fn = cm.sum(1) - tp
        res: Dict[str, float] = {}
        ious, dices = [], []
        for c in range(self.num_classes):
            name = ID_TO_NAME[c]
            d_iou  = tp[c] + fp[c] + fn[c]
            d_dice = 2 * tp[c] + fp[c] + fn[c]
            iou  = float(tp[c] / d_iou)  if d_iou  > 0 else 1.0
            dice = float(2 * tp[c] / d_dice) if d_dice > 0 else 1.0
            res[f"iou_{name}"]  = iou
            res[f"dice_{name}"] = dice
            ious.append(iou); dices.append(dice)
        res["mean_iou"]  = float(np.mean(ious))
        res["mean_dice"] = float(np.mean(dices))
        return res

print("MetricTracker ready.")


---
## 14. Training & Validation Engine with Checkpointing

Features:
- **AMP** (`torch.cuda.amp`) — 2x faster on Kaggle GPU
- **DeepLabV3 output handling** — the model returns `OrderedDict{"out": logits}`;
  the engine extracts `output["out"]` automatically.
- **Checkpoint resume** — saves `checkpoint_latest.pth` every epoch;
  `best_model.pth` only when Validation mIoU improves.


In [ ]:
def extract_logits(model_output) -> torch.Tensor:
    """
    Handle both DeepLabV3 (returns OrderedDict) and U-Net (returns Tensor).
    DeepLabV3 output: {'out': Tensor(B, C, H, W), 'aux': Tensor(B, C, H, W)}
    U-Net output    : Tensor(B, C, H, W)
    """
    if isinstance(model_output, dict):
        return model_output["out"]
    return model_output


def train_one_epoch(model, loader, criterion, optimizer, scaler, device, tracker):
    model.train()
    tracker.reset()
    total_loss = 0.0
    pbar = tqdm(loader, desc="Train", leave=False)
    aux_loss_weight = CONFIG.get("aux_loss_weight", 0.4)
    for imgs, masks in pbar:
        imgs  = imgs.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        if CONFIG["use_amp"] and device.type == "cuda":
            with autocast():
                output = model(imgs)
                if isinstance(output, dict):
                    loss_main, _, _ = criterion(output["out"], masks)
                    if "aux" in output and CONFIG.get("aux_loss", True):
                        loss_aux, _, _ = criterion(output["aux"], masks)
                        loss = loss_main + aux_loss_weight * loss_aux
                    else:
                        loss = loss_main
                    logits = output["out"]
                else:
                    logits = output
                    loss, _, _ = criterion(logits, masks)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            output = model(imgs)
            if isinstance(output, dict):
                loss_main, _, _ = criterion(output["out"], masks)
                if "aux" in output and CONFIG.get("aux_loss", True):
                    loss_aux, _, _ = criterion(output["aux"], masks)
                    loss = loss_main + aux_loss_weight * loss_aux
                else:
                    loss = loss_main
                logits = output["out"]
            else:
                logits = output
                loss, _, _ = criterion(logits, masks)
            loss.backward()
            optimizer.step()
        total_loss += loss.item()
        tracker.update(torch.argmax(logits, 1), masks)
        pbar.set_postfix(loss=f"{loss.item():.4f}")
    return total_loss / max(len(loader), 1), tracker.compute()


def evaluate_epoch(model, loader, criterion, device, tracker):
    model.eval()
    tracker.reset()
    total_loss = 0.0
    with torch.no_grad():
        pbar = tqdm(loader, desc="Eval", leave=False)
        for imgs, masks in pbar:
            imgs  = imgs.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True)
            if CONFIG["use_amp"] and device.type == "cuda":
                with autocast():
                    output = model(imgs)
                    logits = output["out"] if isinstance(output, dict) else output
                    loss, _, _ = criterion(logits, masks)
            else:
                output = model(imgs)
                logits = output["out"] if isinstance(output, dict) else output
                loss, _, _ = criterion(logits, masks)
            total_loss += loss.item()
            tracker.update(torch.argmax(logits, 1), masks)
            pbar.set_postfix(val_loss=f"{loss.item():.4f}")
    return total_loss / max(len(loader), 1), tracker.compute()


def save_checkpoint(state: Dict, is_best: bool):
    latest = os.path.join(CONFIG["checkpoint_dir"], "checkpoint_latest.pth")
    torch.save(state, latest)
    if is_best:
        best = os.path.join(CONFIG["output_dir"], "best_model.pth")
        torch.save(state["model_state_dict"], best)
        print(f"  [*] New best model saved: {best}")


def load_checkpoint(path, model, optimizer, scheduler=None, scaler=None):
    if not os.path.exists(path):
        return 1, 0.0, {}
    print(f"Resuming from: {path}")
    ckpt = torch.load(path, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    if scheduler and "scheduler_state_dict" in ckpt:
        scheduler.load_state_dict(ckpt["scheduler_state_dict"])
    if scaler and "scaler_state_dict" in ckpt:
        scaler.load_state_dict(ckpt["scaler_state_dict"])
    return ckpt.get("epoch", 0) + 1, ckpt.get("best_iou", 0.0), ckpt.get("history", {})

print("Training engine defined (supports both DeepLabV3 dict output and U-Net tensor output).")


---
## 15. Run Model Training

**Differential Learning Rate Strategy:**

| Parameter group | LR | Rationale |
|---|---|---|
| ResNet-50 backbone | `backbone_lr = 1e-5` | Preserve pretrained ImageNet features |
| DeepLabV3 classifier head | `head_lr = 1e-4` | Train new 3-class head faster |
| Auxiliary head | `head_lr = 1e-4` | Consistent with main head |

Both groups use the same `weight_decay` and are stepped together by
`CosineAnnealingLR`. Checkpoint resume is automatic.


In [ ]:
# Differential learning rate optimizer for transfer learning
backbone_params    = list(model.backbone.parameters())
classifier_params  = list(model.classifier.parameters()) + \
                     list(model.aux_classifier.parameters())

optimizer = torch.optim.AdamW([
    {"params": backbone_params,   "lr": CONFIG["backbone_lr"]},
    {"params": classifier_params, "lr": CONFIG["head_lr"]},
], weight_decay=CONFIG["weight_decay"])

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=CONFIG["num_epochs"], eta_min=CONFIG["min_lr"]
)
scaler  = GradScaler(enabled=(CONFIG["use_amp"] and device.type == "cuda"))
tracker = MetricTracker(num_classes=CONFIG["num_classes"])

history = {k: [] for k in [
    "train_loss", "val_loss", "train_mean_iou",
    "val_mean_iou", "val_pothole_iou", "val_crack_iou", "learning_rates"
]}
best_val_iou = 0.0
start_epoch  = 1

resume_path = os.path.join(CONFIG["checkpoint_dir"], "checkpoint_latest.pth")
if os.path.exists(resume_path):
    start_epoch, best_val_iou, saved_hist = load_checkpoint(
        resume_path, model, optimizer, scheduler, scaler)
    if saved_hist:
        history = saved_hist

print(f"Model         : {CONFIG['model_architecture']}")
print(f"Backbone LR   : {CONFIG['backbone_lr']:.0e}")
print(f"Head LR       : {CONFIG['head_lr']:.0e}")
print(f"Epochs        : {CONFIG['num_epochs']}")
print(f"Device        : {device}")
print("-" * 75)

if train_loader is not None and len(train_loader) > 0:
    for epoch in range(start_epoch, CONFIG["num_epochs"] + 1):
        t0 = time.time()
        lr_bb  = optimizer.param_groups[0]["lr"]
        lr_hd  = optimizer.param_groups[1]["lr"]

        tr_loss, tr_m = train_one_epoch(
            model, train_loader, criterion, optimizer, scaler, device, tracker)
        va_loss, va_m = evaluate_epoch(
            model, val_loader, criterion, device, tracker)
        scheduler.step()

        history["train_loss"].append(tr_loss)
        history["val_loss"].append(va_loss)
        history["train_mean_iou"].append(tr_m["mean_iou"])
        history["val_mean_iou"].append(va_m["mean_iou"])
        history["val_pothole_iou"].append(va_m["iou_pothole"])
        history["val_crack_iou"].append(va_m["iou_crack"])
        history["learning_rates"].append(lr_hd)

        is_best = va_m["mean_iou"] > best_val_iou
        if is_best:
            best_val_iou = va_m["mean_iou"]

        save_checkpoint({
            "epoch"               : epoch,
            "model_state_dict"    : model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "scaler_state_dict"   : scaler.state_dict(),
            "best_iou"            : best_val_iou,
            "history"             : history,
            "architecture"        : CONFIG["model_architecture"],
        }, is_best=is_best)

        print(
            f"Epoch [{epoch:02d}/{CONFIG['num_epochs']:02d}] "
            f"({time.time()-t0:.1f}s)  "
            f"Loss [tr={tr_loss:.4f} va={va_loss:.4f}]  "
            f"mIoU [tr={tr_m['mean_iou']:.3f} va={va_m['mean_iou']:.3f}]  "
            f"Pothole IoU={va_m['iou_pothole']:.3f}  "
            f"Crack IoU={va_m['iou_crack']:.3f}  "
            f"LR(bb={lr_bb:.0e} hd={lr_hd:.0e})  "
            f"{'[*] BEST' if is_best else ''}"
        )
else:
    print("[!] Training skipped: DataLoader empty or dataset not attached.")


---
## 16. Training History Visualisation


In [ ]:
def plot_training_history(history: Dict, save_path: str = None):
    if not history or not history.get("train_loss"):
        print("No training history to plot.")
        return
    epochs = range(1, len(history["train_loss"]) + 1)
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    axes[0].plot(epochs, history["train_loss"], "b-o", ms=3, label="Train")
    axes[0].plot(epochs, history["val_loss"],   "r-s", ms=3, label="Val")
    axes[0].set_title("Loss"); axes[0].set_xlabel("Epoch")
    axes[0].legend(); axes[0].grid(True, alpha=0.3)

    axes[1].plot(epochs, history["train_mean_iou"], "b-o", ms=3, label="Train mIoU")
    axes[1].plot(epochs, history["val_mean_iou"],   "g-^", ms=3, label="Val mIoU")
    axes[1].set_title("Mean IoU"); axes[1].set_xlabel("Epoch")
    axes[1].legend(); axes[1].grid(True, alpha=0.3)

    axes[2].plot(epochs, history["val_pothole_iou"], "r-d", ms=3, label="Pothole IoU")
    axes[2].plot(epochs, history["val_crack_iou"],   "y-x", ms=3, label="Crack IoU")
    axes[2].set_title("Class-wise Validation IoU"); axes[2].set_xlabel("Epoch")
    axes[2].legend(); axes[2].grid(True, alpha=0.3)

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=200, bbox_inches="tight")
        print(f"Curves saved: {save_path}")
    plt.show()

plot_training_history(
    history,
    save_path=os.path.join(CONFIG["output_dir"], "training_curves.png")
)


---
## 17. Test Set Evaluation

Load the best-checkpoint weights and evaluate on the held-out test split.

> Results are based on polygon-rasterised approximate masks. Reported IoU/Dice
> values reflect agreement between model predictions and those polygon masks,
> not against precise pixel-level ground truth.


In [ ]:
best_model_path = os.path.join(CONFIG["output_dir"], "best_model.pth")
test_results    = {}

if os.path.exists(best_model_path) and test_loader is not None:
    print(f"Loading best weights: {best_model_path}")
    model.load_state_dict(torch.load(best_model_path, map_location=device))

    test_loss, test_results = evaluate_epoch(
        model, test_loader, criterion, device, tracker)

    print("\n" + "=" * 60)
    print("FINAL TEST SET RESULTS  (polygon-mask approximate annotations)")
    print("=" * 60)
    print(f"  Test Loss       : {test_loss:.4f}")
    print(f"  Mean IoU (mIoU) : {test_results['mean_iou']:.4f}")
    print(f"  Mean Dice       : {test_results['mean_dice']:.4f}")
    print("-" * 60)
    for cls_id, name in ID_TO_NAME.items():
        iou  = test_results.get(f"iou_{name}",  0)
        dice = test_results.get(f"dice_{name}", 0)
        print(f"  {name:12s}  IoU={iou:.4f}  Dice={dice:.4f}")
    print("=" * 60)
else:
    print("Test evaluation will run automatically after training produces best_model.pth.")


---
## 18. Prediction Visualisation on Unseen Test Images

4-panel view per sample:
1. Original image
2. Ground truth (polygon-rasterised approximate mask)
3. Model prediction
4. Prediction overlay


In [ ]:
def visualize_test_predictions(model, dataset, device, n: int = 4):
    if dataset is None or len(dataset) == 0:
        print("No test dataset.")
        return
    model.eval()
    n = min(n, len(dataset))
    indices = random.sample(range(len(dataset)), n)
    mean = np.array(CONFIG["img_mean"])
    std  = np.array(CONFIG["img_std"])

    fig, axes = plt.subplots(n, 4, figsize=(20, 5 * n))
    if n == 1:
        axes = np.expand_dims(axes, 0)
    fig.suptitle(
        "Test Predictions  |  GT = polygon-rasterised approximate masks",
        fontsize=11)

    with torch.no_grad():
        for i, idx in enumerate(indices):
            img_t, mask_t = dataset[idx]
            raw_out  = model(img_t.unsqueeze(0).to(device))
            logits   = extract_logits(raw_out)
            pred     = torch.argmax(logits, 1).squeeze(0).cpu().numpy()
            gt       = mask_t.numpy()
            img_vis  = np.clip((img_t.permute(1,2,0).numpy() * std + mean) * 255,
                               0, 255).astype(np.uint8)

            axes[i,0].imshow(img_vis)
            axes[i,0].set_title("Original"); axes[i,0].axis("off")
            axes[i,1].imshow(mask_to_color_image(gt))
            axes[i,1].set_title("GT (polygon approx)"); axes[i,1].axis("off")
            axes[i,2].imshow(mask_to_color_image(pred))
            axes[i,2].set_title("Prediction"); axes[i,2].axis("off")
            axes[i,3].imshow(overlay_mask_on_image(img_vis, pred, 0.55))
            axes[i,3].set_title("Overlay"); axes[i,3].axis("off")

    plt.tight_layout()
    out_path = os.path.join(CONFIG["output_dir"], "test_predictions_sample.png")
    plt.savefig(out_path, dpi=200, bbox_inches="tight")
    print(f"Saved: {out_path}")
    plt.show()

if os.path.exists(best_model_path) and test_ds is not None:
    visualize_test_predictions(model, test_ds, device, n=4)
else:
    print("Visualisation will run once best_model.pth is produced.")


---
## 19. Production Artifact Export for FastAPI

Exports 4 files to `/kaggle/working/roadvision_artifacts/` and bundles them
into `roadvision_model_artifacts.zip` for one-click download:

| File | Purpose |
|---|---|
| `best_model.pth` | DeepLabV3-ResNet50 `state_dict` |
| `model_config.json` | Architecture + preprocessing config (FastAPI reads this) |
| `class_mapping.json` | Class IDs, names, colours |
| `training_metrics.json` | IoU, Dice, epoch counts |


In [ ]:
def export_production_artifacts():
    out = Path(CONFIG["output_dir"])
    out.mkdir(parents=True, exist_ok=True)

    # 1. model_config.json  — read by FastAPI backend at startup
    cfg = {
        "model_name"       : "RoadVision_DeepLabV3_ResNet50",
        "architecture"     : CONFIG["model_architecture"],    # "DeepLabV3_ResNet50"
        "backbone"         : CONFIG["backbone"],               # "resnet50"
        "pretrained_weights": CONFIG["pretrained_weights"],    # for reproducibility
        "in_channels"      : CONFIG["in_channels"],
        "num_classes"      : CONFIG["num_classes"],
        "aux_loss"         : CONFIG.get("aux_loss", True),
        "aux_loss_weight"  : CONFIG.get("aux_loss_weight", 0.4),
        "input_resolution" : {
            "height": CONFIG["image_size"][0],
            "width" : CONFIG["image_size"][1],
        },
        "normalization"    : {
            "mean": CONFIG["img_mean"],
            "std" : CONFIG["img_std"],
        },
        "training_hyperparameters": {
            "batch_size"   : CONFIG["batch_size"],
            "backbone_lr"  : CONFIG["backbone_lr"],
            "head_lr"      : CONFIG["head_lr"],
            "optimizer"    : "AdamW (differential LR)",
            "loss_function": "WeightedCE+Dice",
            "class_weights": CONFIG["class_weights"],
        },
        "dataset"          : CONFIG["dataset_name"],
        "annotation_type"  : "polygon_rasterised_approximate",
        "export_timestamp" : time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        "framework"        : f"PyTorch {torch.__version__}",
        "torchvision"      : torchvision.__version__,
    }
    (out / "model_config.json").write_text(json.dumps(cfg, indent=2))
    print("[1/4] model_config.json")

    # 2. class_mapping.json
    cls_map = {
        "classes"          : CLASS_INFO,
        "id_to_name"       : ID_TO_NAME,
        "name_to_id"       : NAME_TO_ID,
        "dataset_to_model" : DATASET_TO_MODEL_CLASS,
        "num_classes"      : CONFIG["num_classes"],
        "damage_classes"   : ["pothole", "crack"],
        "excluded_classes" : ["manhole"],
    }
    (out / "class_mapping.json").write_text(json.dumps(cls_map, indent=2))
    print("[2/4] class_mapping.json")

    # 3. training_metrics.json
    metrics = {
        "best_validation_miou"    : float(best_val_iou),
        "test_metrics"            : test_results,
        "epochs_completed"        : len(history.get("train_loss", [])),
        "total_epochs_configured" : CONFIG["num_epochs"],
        "device_used"             : str(device),
        "annotation_note"         : (
            "Metrics computed against polygon-rasterised approximate masks, "
            "not precise pixel-level ground truth."
        ),
    }
    (out / "training_metrics.json").write_text(json.dumps(metrics, indent=2))
    print("[3/4] training_metrics.json")

    # 4. best_model.pth
    best_path = out / "best_model.pth"
    if not best_path.exists():
        torch.save(model.state_dict(), best_path)
    sz = best_path.stat().st_size / 1e6
    print(f"[4/4] best_model.pth  ({sz:.1f} MB)")

    # Zip bundle for one-click Kaggle download
    zip_p = Path("/kaggle/working/roadvision_model_artifacts.zip")
    with zipfile.ZipFile(zip_p, "w", zipfile.ZIP_DEFLATED) as zf:
        for f in out.glob("*"):
            if f.is_file():
                zf.write(f, arcname=f.name)
    print(f"\nDownload bundle: {zip_p}  ({zip_p.stat().st_size/1e6:.1f} MB)")

export_production_artifacts()


---
## 20. FastAPI Integration Verification

Proves the exported artifacts are self-contained: load `model_config.json`,
re-instantiate **DeepLabV3-ResNet50** without pretrained weights (we load
`best_model.pth` state dict instead), run a forward pass, and confirm output
shape and class labels are correct.

This is exactly what the FastAPI inference server will do at startup.


In [ ]:
def verify_fastapi_readiness():
    out = Path(CONFIG["output_dir"])

    with open(out / "model_config.json") as f:
        cfg = json.load(f)
    with open(out / "class_mapping.json") as f:
        cls_map = json.load(f)

    # Re-instantiate from config with matching aux_loss (no pretrained weights — load state_dict instead)
    aux_loss_enabled = cfg.get("aux_loss", True)
    serve_model = deeplabv3_resnet50(weights=None,
                                     num_classes=cfg["num_classes"],
                                     aux_loss=aux_loss_enabled).to(device)
    serve_model.load_state_dict(
        torch.load(out / "best_model.pth", map_location=device))
    serve_model.eval()

    h = cfg["input_resolution"]["height"]
    w = cfg["input_resolution"]["width"]
    dummy = torch.randn(1, 3, h, w, device=device)

    with torch.no_grad():
        raw_out = serve_model(dummy)
        logits  = extract_logits(raw_out)
        probs   = F.softmax(logits, 1)
        pred    = torch.argmax(probs, 1).squeeze(0).cpu().numpy()

    total_p = sum(p.numel() for p in serve_model.parameters())

    print("[OK] model_config.json and class_mapping.json loaded")
    print(f"[OK] DeepLabV3-ResNet50 re-instantiated ({total_p:,} params)")
    print(f"[OK] best_model.pth weights loaded cleanly")
    print(f"[OK] Input  : {tuple(dummy.shape)}")
    print(f"[OK] Logits : {tuple(logits.shape)}")
    print(f"[OK] Mask   : {pred.shape}  dtype={pred.dtype}")
    print(f"[OK] Classes: {cls_map['damage_classes']}  excluded: {cls_map['excluded_classes']}")
    print(f"[OK] Architecture in config: {cfg['architecture']}")
    print(f"[OK] Backbone in config    : {cfg['backbone']}")
    print("\nFastAPI Integration Verification: ALL CHECKS PASSED")

verify_fastapi_readiness()
